# 音のプログラミング 第1回: 基本概念とサイン波

**学習目標:**
- 音とプログラミングの関係を理解する
- サイン波（正弦波）を生成する
- デジタル音声とサンプリングの仕組みを知る
- 周波数と音の高さの関係を体験する
- 初めての音を生成する

**所要時間:** 90分

## 🛠️ 環境セットアップ

In [ ]:
# 🛠️ 環境セットアップ
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colab環境で実行中...")
except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")

if IN_COLAB:
    print("🔧 Google Colab環境を設定中...")
    !pip install japanize-matplotlib
    !git clone https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
    import japanize_matplotlib
else:
    print("🔧 ローカル環境を設定中...")
    sys.path.append('..')
    import platform
    if platform.system() == 'Darwin':
        plt.rcParams['font.family'] = 'Hiragino Sans'
    else:
        plt.rcParams['font.family'] = 'Meiryo'

print("✅ セットアップ完了")

## 🛠️ 今回使用するライブラリ

In [ ]:
from audio_lib import sine_wave, save_audio, AudioSignal
from audio_lib import sawtooth_wave
from audio_lib.synthesis.note_utils import note_to_frequency, note_name_to_number
from audio_lib.notebook import play_sound, plot_waveform

## 🎵 音とは何か？

音とは**空気の振動**です。この振動を数式で表すと：

$$y = \sin(2\pi \times \text{周波数} \times t)$$

- **周波数 (Hz)**: 1秒間に振動する回数 → 音の高さ
- **振幅**: 振動の大きさ → 音の大きさ（音量）

## 🎯 実習1: 最初のサイン波を作ろう

In [ ]:
# 440Hz（ラの音）のサイン波を1秒間生成
signal = sine_wave(frequency=440, duration=1.0)

print(f"データの長さ: {signal.num_samples} サンプル")
print(f"最大振幅: {max(signal.data):.4f}")
print(f"最小振幅: {min(signal.data):.4f}")

In [ ]:
# 波形を表示（最初の0.01秒 = 約4.4周期分）
plot_waveform(signal, duration=0.01, title="440Hz サイン波")

きれいな波形が見えますね。これがサイン波（正弦波）です。

In [ ]:
# 音を再生してみよう
display(play_sound(signal, "440Hz サイン波（ラの音）"))

440Hzはラ（A4）の音です。とてもシンプルで純粋な音がします。

## 🎵 デジタル音声とサンプリング

コンピュータで音を扱うには、連続的な波形を一定間隔で「測定」する必要があります。これを**サンプリング**といいます。

- **サンプリングレート**: 1秒間に何回測定するか（CDは44100回/秒）
- **ナイキストの定理**: サンプリングレートの半分の周波数までしか正確に記録できない
- なぜ44100Hz？→ 人間の可聴域（〜20000Hz）を超える22050Hzまでカバー

| 用語 | 説明 | 例 |
|------|------|-----|
| サンプリングレート | 1秒間のサンプル数 | 44100 Hz |
| ナイキスト周波数 | 記録可能な最高周波数 | 22050 Hz |
| ビット深度 | 1サンプルの精度 | 16bit, 24bit |

In [ ]:
# サンプリングレートの違いを体験しよう
# CD品質（44100Hz）vs 電話品質（8000Hz）

saw_cd = sawtooth_wave(440, 1.5, sample_rate=44100)
saw_tel = sawtooth_wave(440, 1.5, sample_rate=8000)

display(play_sound(saw_cd, "CD品質 (44100Hz) のこぎり波"))
display(play_sound(saw_tel, "電話品質 (8000Hz) のこぎり波"))

電話品質の音はこもって聞こえます。
サンプリングレート8000Hzでは4000Hz以上の倍音が記録できないため、音色が変わるのです。

In [ ]:
# ナイキスト周波数違反の例：5000Hzの音を8000Hzでサンプリング
high_cd = sine_wave(5000, 0.5, sample_rate=44100)
high_tel = sine_wave(5000, 0.5, sample_rate=8000)

display(play_sound(high_cd, "5000Hz（CD品質: 正確に再現）"))
display(play_sound(high_tel, "5000Hz（電話品質: エイリアシング発生）"))

5000Hzは8000Hzの半分（4000Hz）を超えています。
そのため**エイリアシング**が発生し、元とは異なる音に聞こえます。

## 🎯 実習2: 周波数を変えて音程を体験しよう

In [ ]:
# 4つの異なる周波数を聞いてみよう
frequencies = [220, 440, 880, 1760]

for freq in frequencies:
    sig = sine_wave(freq, 1.5)
    display(play_sound(sig, f"{freq}Hz サイン波"))

周波数が2倍になると1オクターブ上がります:
- 220Hz → 440Hz → 880Hz → 1760Hz

これが「オクターブ」の関係です。

## 🎯 実習3: 音量を変えてみよう

振幅（amplitude）を変えると音量が変わります。

In [ ]:
# 同じ440Hzで音量を変える
volumes = [0.1, 0.3, 0.8, 1.0]

for vol in volumes:
    sig = sine_wave(440, 1.0)
    sig_vol = sig * vol
    display(play_sound(sig_vol, f"440Hz 音量={vol}"))

振幅が大きいほど音が大きくなります。

## 🎯 実習4: 音名と周波数の関係

In [ ]:
# C4（ド）からC5（高いド）までの音名・周波数対応表
note_names = ["C4", "D4", "E4", "F4", "G4", "A4", "B4", "C5"]
jp_names = ["ド", "レ", "ミ", "ファ", "ソ", "ラ", "シ", "ド"]

print("音名  | 日本名 | MIDI番号 | 周波数 (Hz)")
print("-" * 45)
for name, jp in zip(note_names, jp_names):
    midi_num = note_name_to_number(name)
    freq = note_to_frequency(midi_num)
    print(f" {name}  |  {jp:2s}  |    {midi_num}    | {freq:.2f}")

In [ ]:
# ドレミファソラシドを演奏
scale_notes = ["C4", "D4", "E4", "F4", "G4", "A4", "B4", "C5"]

for name in scale_notes:
    midi_num = note_name_to_number(name)
    freq = note_to_frequency(midi_num)
    sig = sine_wave(freq, 0.6)
    display(play_sound(sig, f"{name}"))

ドレミファソラシドの音階が完成しました！

## 🏆 チャレンジ課題

In [ ]:
# チャレンジ1: 好きな周波数で音を作ってみよう
my_freq = 500  # ← 好きな値に変更してみよう
my_duration = 2.0

my_sound = sine_wave(my_freq, my_duration)
display(play_sound(my_sound, f"マイサウンド ({my_freq}Hz)"))

In [ ]:
# チャレンジ2: 2つの音を同時に鳴らしてハーモニーを作ろう
signal1 = sine_wave(440, 2.0)   # ラ
signal2 = sine_wave(554, 2.0)   # ド#

harmony = (signal1 + signal2) * 0.5
display(play_sound(harmony, "ハーモニー (440Hz + 554Hz)"))

2つの音を足し合わせると和音（ハーモニー）になります。

## 📚 今日のまとめ

### 学んだこと
1. 音は空気の振動であり、サイン波は最も基本的な波形
2. **周波数**（Hz）が音の高さを決める
3. **振幅**が音の大きさを決める
4. **サンプリング**でアナログ音声をデジタルに変換する
5. **ナイキストの定理**: サンプリングレートの半分の周波数が限界
6. 周波数が2倍になると1オクターブ上がる
7. 音名（C4, A4など）はMIDI番号と対応している
8. 複数のサイン波を足すとハーモニーが生まれる

### 使ったライブラリ関数
- `sine_wave(freq, duration)` — サイン波の生成
- `play_sound(signal, title)` — 音声の再生
- `plot_waveform(signal, duration)` — 波形の表示
- `note_to_frequency(midi_num)` — MIDI番号→周波数
- `note_name_to_number(name)` — 音名→MIDI番号

### 次回予告
第2回では**エンベロープ**と**ADSR**を学び、音にリアルな時間変化を付けます。